In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 6
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .replace("<s>", "")
                .replace("</s>", "")
                .replace("<pad>", "")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [5]:
import numpy as np
import pickle


bpe_path = "/kaggle/input/datasets/girieshpassya/tokenizer-results/project_bpe_16k.pkl"
train_path = "/kaggle/input/datasets/girieshpassya/tokenizer-results/train_tokens1 (2).npy"
val_path = "/kaggle/input/datasets/girieshpassya/tokenizer-results/val_tokens1 (1).npy"
# train2_path = "/kaggle/input/datasets/girieshpassya/next-200/train_tokens2.npy"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    bpe = pickle.load(f)

# Load token arrays
train_tokens = np.load(train_path, mmap_mode="r")
val_tokens = np.load(val_path, mmap_mode="r")
# train2_tokens = np.load(train2_path, mmap_mode = "r")
# train_tokens =np.concatenate([train1_tokens, train2_tokens])

print("BPE vocab size:", len(bpe.itos))
print("Train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))


BPE vocab size: 7999
Train tokens: 199999998
Validation tokens: 5000000


In [6]:
train_tokens[0]

np.uint16(2)

In [7]:

import os
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0
BOS_ID =2
EOS_ID = 3

BLOCK_SIZE =  1023
MIN_BLOCK = 64
def build_block_index(ids, index_path, block_size=BLOCK_SIZE):
    document_start = 0
    block_count = 0

    with open(index_path, "wb") as out:

        def write_document(doc_start, doc_end, has_eos):
            nonlocal block_count

            start, end = doc_start, doc_end

            if start < end and int(ids[start]) == BOS_ID:
                start += 1
            if start < end and int(ids[end - 1]) == EOS_ID:
                end -= 1

            for pos in range(start, end, block_size):
                length = min(block_size, end - pos)
                if length < MIN_BLOCK:
                    continue

                real_end = int(has_eos and pos + length == end)
                np.asarray(
                    [pos, length, real_end], dtype=np.int64
                ).tofile(out)
                block_count += 1

        for base in range(0, len(ids), 1_000_000):
            positions = (
                np.flatnonzero(
                    ids[base:base + 1_000_000] == EOS_ID
                ) + base
            )

            for eos_pos in positions:
                write_document(
                    document_start, int(eos_pos) + 1, True
                )
                document_start = int(eos_pos) + 1

        if document_start < len(ids):
            write_document(document_start, len(ids), False)

    if block_count == 0:
        raise ValueError("No usable blocks found.")

    return block_count


class EncoderDecoderDataset(Dataset):
    def __init__(self, ids, index_path, training):
        self.ids = ids  # Pass the array directly
        self.index = np.memmap(
            index_path, dtype=np.int64, mode="r"
        ).reshape(-1, 3)
        self.training = training

    def __len__(self):
        return len(self.index)

    # Keep your existing __getitem__ method unchanged.
       
    def __getitem__(self, i):
        start, length, real_end = map(int, self.index[i])
    
        tokens = torch.tensor(
            np.asarray(self.ids[start:start + length], dtype=np.int64),
            dtype=torch.long,
        )

        if self.training:
            # Always leave at least one token for the target.
            split = int(torch.randint(64, length + 1, ()).item())
            split = min(split, length - 1)
        else:
            # Fixed validation split, leaving at least one target token.
            split = max(64, int(length * 0.75))
            split = min(split, length - 1)
    
        src = torch.cat([
            torch.tensor([BOS_ID]),
            tokens[:split],
        ])
    
        suffix = tokens[split:]
    
        if real_end:
            suffix = torch.cat([
                suffix,
                torch.tensor([EOS_ID]),
            ])
    
        decoder_input = torch.cat([
            torch.tensor([BOS_ID]),
            suffix[:-1],
        ])
    
        return src, decoder_input, suffix



def collate_prefix(items):
    src, dec, labels = zip(*items)

    return {
        "src_ids": pad_sequence(
            src, batch_first=True, padding_value=PAD_ID
        ),
        "decoder_ids": pad_sequence(
            dec, batch_first=True, padding_value=PAD_ID
        ),
        "labels": pad_sequence(
            labels, batch_first=True, padding_value=-1
        ),
    }
train_index_path = os.path.join(OUTPUT_DIR, "train_blocks.int64")
val_index_path = os.path.join(OUTPUT_DIR, "val_blocks.int64")

print("Train blocks:", build_block_index(train_tokens, train_index_path))
print("Validation blocks:", build_block_index(val_tokens, val_index_path))

train_dataset = EncoderDecoderDataset(train_tokens, train_index_path, training=True)
val_dataset = EncoderDecoderDataset(val_tokens, val_index_path, training=False)


Train blocks: 358960
Validation blocks: 10442


In [8]:
len(train_dataset)//BATCH_SIZE

22435

In [9]:
src, decoder_input, labels = train_dataset[0]

print("Encoder length:", len(src))
print("Decoder input length:", len(decoder_input))
print("Labels length:", len(labels))

print("Max encoder length:", max(len(train_dataset[i][0])
                                  for i in range(min(358960, len(train_dataset)))))

Encoder length: 167
Decoder input length: 857
Labels length: 857
Max encoder length: 1023


In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_prefix,
   
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_prefix,
   
)

In [11]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [12]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None, is_causal=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.is_causal = is_causal
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, padding_mask = None, past_key_value=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        # Apply RoPE using absolute positions, before caching keys.
        if self.use_rope:
            cos = self.rope_cos[position_offset:position_offset + T]
            sin = self.rope_sin[position_offset:position_offset + T]
            q = apply_rope(
                q, cos, sin
            )
            k = apply_rope(
                k, cos, sin
            )

        # Append the new K/V to the cached prefix.
        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present = (k, v) if use_cache else None
        key_len = k.size(2)

        # Expand grouped KV heads for attention.
        if self.n_rep > 1:
            k_attn = k.repeat_interleave(self.n_rep, dim=1)
            v_attn = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_attn, v_attn = k, v

        scores = (q @ k_attn.transpose(-2, -1)) * self.attention_scale

        if self.is_causal:
            # During cached decoding, query positions start at position_offset.
            q_positions = torch.arange(
                position_offset, position_offset + T, device=x.device
            )
            k_positions = torch.arange(key_len, device=x.device)
            causal = k_positions[None, :] <= q_positions[:, None]
            scores = scores.masked_fill(
                ~causal[None, None, :, :], float("-inf")
            )

        if padding_mask is not None:
            # Padding mask describes keys in the complete prefix.
            if padding_mask.shape != (B, key_len):
                raise ValueError(
                    f"Expected padding_mask {(B, key_len)}, "
                    f"got {tuple(padding_mask.shape)}"
                )
            padding_mask = padding_mask.to(device=x.device, dtype=torch.bool)
            if not padding_mask.any(dim=-1).all():
                raise ValueError("sequence does not have valid key")
            scores = scores.masked_fill(
                ~padding_mask[:, None, None, :], float("-inf")
            )

        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v_attn
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        if use_cache:
            return out, present
        return out


In [13]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_inputs, source_padding_mask=None, past_key_value=None, use_cache=False):
        B, T, C = x.shape
        B_enc, S, _ = encoder_inputs.shape
        if B != B_enc:
            raise ValueError("Encoder and decoder batch sizes must match.")

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        # Reuse encoder projections after the first decoding step.
        if past_key_value is None:
            k = self.k_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
            v = self.v_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
        else:
            k, v = past_key_value

        present = (k, v) if use_cache else None

        if self.n_rep > 1:
            k_attn = k.repeat_interleave(self.n_rep, dim=1)
            v_attn = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_attn, v_attn = k, v

        scores = (q @ k_attn.transpose(-2, -1)) * self.attention_scale

        if source_padding_mask is not None:
            if source_padding_mask.shape != (B, S):
                raise ValueError("source_padding_mask must have shape (B, S).")
            source_padding_mask = source_padding_mask.to(
                device=x.device, dtype=torch.bool
            )
            if not source_padding_mask.any(dim=-1).all():
                raise ValueError("sequence does not have valid tokens")
            scores = scores.masked_fill(
                ~source_padding_mask[:, None, None, :], float("-inf")
            )

        weights = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = weights @ v_attn
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        if use_cache:
            return out, present
        return out



In [14]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE



class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, padding_mask=None):
        x = x + self.attn(self.ln1(x),padding_mask=padding_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        
        

    def forward(self, idx: torch.Tensor, padding_mask=None) -> torch.Tensor:
        B, T = idx.shape
        
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError("padding_mask must have shape (B, T)")
            padding_mask = padding_mask.to(
                device=idx.device, dtype=torch.bool
            )
        x =self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
          x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
          x=block(x, padding_mask=padding_mask)
        x = self.ln_f(x)
        return x # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")



In [15]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F





class DecoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.self_attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale, is_causal=True )
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn =  CrossAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = DecoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    # def forward(self, x, encoder_inputs,  target_padding_mask=None, source_padding_mask=None):
    #     x = x + self.self_attn(self.ln1(x), padding_mask=target_padding_mask)
    #     x = x + self.cross_attn( self.ln2(x), encoder_inputs, source_padding_mask=source_padding_mask)
    #     x = x + self.ffn(self.ln3(x))
    #     return x   # unchanged from Question 1
    def forward(self, x, encoder_inputs,  target_padding_mask=None, source_padding_mask=None, past_key_value=None, use_cache=False, position_offset=0):
        if past_key_value is None:
            past_self = past_cross = None
        else:
            past_self, past_cross = past_key_value
    
        if use_cache:
            attn, present_self = self.self_attn(
                self.ln1(x),
                padding_mask=target_padding_mask,
                past_key_value=past_self,
                use_cache=True,
                position_offset=position_offset,
            )
        else:
            attn = self.self_attn(
                self.ln1(x), padding_mask=target_padding_mask
            )
            present_self = None

        x = x + attn
    
        if use_cache:
            cross, present_cross = self.cross_attn(
                self.ln2(x),
                encoder_inputs,
                source_padding_mask=source_padding_mask,
                past_key_value=past_cross,
                use_cache=True,
            )
        else:
            cross = self.cross_attn(
                self.ln2(x), encoder_inputs,
                source_padding_mask=source_padding_mask,
            )
            present_cross = None
    
        x = x + cross
        x = x + self.ffn(self.ln3(x))
    
        if use_cache:
            return x, (present_self, present_cross)
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor, encoder_inputs:torch.Tensor, target_padding_mask=None, source_padding_mask=None,past_key_values=None, use_cache=False) -> torch.Tensor:
        B, T = idx.shape

        if past_key_values is not None and len(past_key_values) != len(self.blocks):
            raise ValueError("Cache must contain one entry per decoder layer.")
        past_len = (
            0 if past_key_values is None
            else past_key_values[0][0][0].size(2)
        )
        total_len = past_len + T
        if total_len > self.cfg.max_seq_len:
            raise ValueError(
                f"Decoder length {total_len} exceeds "
                f"max_seq_len={self.cfg.max_seq_len}. "
                "Reset or trim the KV cache."
            )
        # Validate the complete decoder key mask.
        if target_padding_mask is not None:
            if target_padding_mask.shape != (B, total_len):
                raise ValueError(
                    f"Expected target mask {(B, total_len)}, "
                    f"got {tuple(target_padding_mask.shape)}"
                )
    
            target_padding_mask = target_padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )
            if not target_padding_mask.any(dim=-1).all():
                raise ValueError(
                    "Every decoder sequence needs at least one valid token."
                )

        x = self.tok_emb(idx)
    
        if self.cfg.pos_emb_type == "sinusoidal":
            end_pos = past_len + T
            if end_pos > self.pos_emb.size(0):
                raise ValueError("Increase cfg.max_seq_len for this sequence.")
            x = x + self.pos_emb[past_len:end_pos].unsqueeze(0)
    
        presents = []
    
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
    
            if use_cache:
                x, present = block(
                    x, encoder_inputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                    past_key_value=past,
                    use_cache=True,
                    position_offset=past_len,
                )
                presents.append(present)
            else:
                x = block(
                    x, encoder_inputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                )
    
        logits = self.head(self.ln_f(x))
    
        if use_cache:
            return logits, tuple(presents)
        return logits
    
            # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")
    


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class EncoderDecoderTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        self.encoder = EncoderLM(cfg)
        self.decoder = DecoderLM(cfg)

    def forward(self, src, tgt, source_padding_mask=None, target_padding_mask=None,past_key_values=None,use_cache=False):
        """
        src: (B, S) source token IDs
        tgt: (B, T) decoder input token IDs

        source_padding_mask: (B, S), True = valid token
        target_padding_mask: (B, T), True = valid token

        Returns:
            logits: (B, T, vocab_size)
        """

        # 1. Encode source tokens
        encoder_outputs = self.encoder(
                            src,
                            padding_mask=source_padding_mask,
                        )

        # 2. Decode target tokens using encoder outputs
        logits = self.decoder(
                    tgt,
                    encoder_outputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                )
        if use_cache:
           logits, present_key_values = result
           return logits, present_key_values

        return logits

In [17]:
@torch.no_grad()
def perplexity_of_loader(model, val_loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in val_loader:
        src = batch["src_ids"].to(device)
        dec = batch["decoder_ids"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(
                src,
                dec,
                source_padding_mask=src.ne(PAD_ID),
                target_padding_mask=dec.ne(PAD_ID),
            )

        if labels.numel() == 0:
            raise ValueError(
                f"Empty labels in validation batch. "
                f"src={src.shape}, dec={dec.shape}"
            )

        if logits.shape[:2] != labels.shape:
            raise ValueError(
                f"Shape mismatch: logits={logits.shape}, "
                f"labels={labels.shape}"
            )

        loss_sum = F.cross_entropy(
            logits.float().reshape(-1, cfg.vocab_size),
            labels.reshape(-1),
            ignore_index=-1,
            reduction="sum",
        )

        total_loss += loss_sum.item()
        total_tokens += labels.ne(-1).sum().item()

    if total_tokens == 0:
        raise ValueError("Validation set contains no valid target tokens.")

    return math.exp(total_loss / total_tokens)

In [18]:

import math
import torch
import time
import torch.nn.functional as F
from itertools import cycle
def get_lr(step, max_steps, peak_lr, warmup, min_ratio=0.1):
    if step < warmup:
        return peak_lr * (step + 1) / warmup

    p = min(1.0, (step - warmup) / max(1, max_steps - warmup))

    return peak_lr * (
        min_ratio
        + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p))
    )

def train(
    model,
    train_loader,
    val_loader,
    optimizer,
    total_steps=1000,
    eval_every=100,
    peak_lr=3e-4,
    warmup = 120,
    save_path="/kaggle/working/encoder_decoder_best.pt",
):
    best_ppl = float("inf")
    best_state = None
    history = []
    model.train()
    start = time.time()
    scaler = torch.amp.GradScaler("cuda")
    train_iter = iter(train_loader)
    for step in range(1, total_steps + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        lr = get_lr(
            step-1,
            max_steps=total_steps,
            peak_lr=peak_lr,
            warmup=warmup,
        )
    
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        src = batch["src_ids"].to(device)
        dec = batch["decoder_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
            logits = model(
                src,
                dec,
                source_padding_mask=src.ne(PAD_ID),
                target_padding_mask=dec.ne(PAD_ID),
            )

        loss_sum = F.cross_entropy(
            logits.float().reshape(-1, VOCAB_SIZE),
            labels.reshape(-1),
            ignore_index=-1,
            reduction="sum",
        )

        valid_tokens = labels.ne(-1).sum()
        loss = loss_sum / valid_tokens.clamp_min(1)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gn = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
        scaler.step(optimizer)
        scaler.update()
        if step % 100 == 0:
            print(
                f"Step {step}/{total_steps} | "
                f"Train Loss: {loss.item():.4f} | "
                f"Time   : {(time.time()-start)/60} in mins"
            )
        if step % eval_every == 0:
            train_loss = loss.item()
            

            val_ppl = perplexity_of_loader(model, val_loader)

            print(
                f"Step {step}/{total_steps} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val PPL: {val_ppl:.2f}  | "
                f"Time   : {(time.time()-start)/60} in mins"
            )

            history.append({
                "step": step,
                "train_loss": train_loss,
                "val_ppl": val_ppl,
                "elapsed_time":(time.time()-start)/60
            })

            if val_ppl < best_ppl:
                best_ppl = val_ppl
                best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                torch.save({
                    "step": step,
                    "model_state_dict": best_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_ppl": best_ppl
                }, save_path)

                print("Saved best checkpoint.")
            model.train()
    if best_state is not None:
      model.load_state_dict(best_state)
    return  history, best_ppl

In [19]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)
model = EncoderDecoderTransformer(cfg)
model = torch.nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
model.to(device)
decay    = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=3e-4, betas=(0.9, 0.95), eps=1e-8,
)


In [20]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model))

total parameters :  77486080


In [21]:
 history, best_ppl = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    total_steps=22500,
    eval_every=1000,
    peak_lr=3e-4,
    warmup=300,
)

Step 100/22500 | Train Loss: 30.4648 | Time   : 1.2928803245226541 in mins
Step 200/22500 | Train Loss: 21.2012 | Time   : 2.5588708241780598 in mins
Step 300/22500 | Train Loss: 14.7530 | Time   : 3.8341835816701253 in mins
Step 400/22500 | Train Loss: 11.1828 | Time   : 5.166970614592234 in mins
Step 500/22500 | Train Loss: 9.8689 | Time   : 6.453291348616282 in mins
Step 600/22500 | Train Loss: 8.1318 | Time   : 7.711136543750763 in mins
Step 700/22500 | Train Loss: 7.3058 | Time   : 8.995594783624012 in mins
Step 800/22500 | Train Loss: 7.1180 | Time   : 10.292446057001749 in mins
Step 900/22500 | Train Loss: 5.8133 | Time   : 11.604536207516988 in mins
Step 1000/22500 | Train Loss: 5.7607 | Time   : 12.90611214240392 in mins
Step 1000/22500 | Train Loss: 5.7607 | Val PPL: 59.69  | Time   : 14.417307456334433 in mins
Saved best checkpoint.
Step 1100/22500 | Train Loss: 5.4304 | Time   : 15.67736298640569 in mins
Step 1200/22500 | Train Loss: 6.7141 | Time   : 16.965556621551514 in 

In [22]:
best_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }
torch.save({
    "history":history,
    "model_state_dict": best_state,
    "optimizer_state_dict": optimizer.state_dict(),
    "best_ppl": best_ppl
}, "encoder_decoder_final.pt")

In [23]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:")
print(test_file_path)
with open(test_file_path,"r", encoding="utf-8") as f:
    test_data = json.load(f)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [24]:
converter = ToTokensConverter(bpe.itos)

In [25]:

import torch


@torch.no_grad()
def generate_tokens(
    model,
    text,
    max_new_words=128,
    temperature=0.8,
    mode="temperature",
    top_k=50,
    device=device,
):
    model.eval()
    max_seq_len = model.cfg.max_seq_len
    if mode not in {"greedy", "temperature", "top_k"}:
        raise ValueError(
            "mode must be 'greedy', 'temperature', or 'top_k'"
        )

    if mode != "greedy" and temperature <= 0:
        raise ValueError("temperature must be positive.")

    # Encode source
    src_token_ids = converter.to_tokens([text])[0]

    if hasattr(src_token_ids, "tolist"):
        src_token_ids = src_token_ids.tolist()

    src_token_ids = src_token_ids[-max_seq_len:]

    if not src_token_ids:
        raise ValueError("Input text produced no tokens.")

    src_ids = torch.tensor(
        [src_token_ids], dtype=torch.long, device=device
    )

    source_padding_mask = torch.ones_like(
        src_ids, dtype=torch.bool
    )

    # Encode source once
    encoder_outputs = model.encoder(
        src_ids,
        padding_mask=source_padding_mask,
    )

    # Decoder starts with the learned tokenizer's BOS ID.
    # This assumes BOS=2, EOS=3, PAD=0 in your vocabulary.
    decoder_context = [2]
    generated = []
    word_count = 0
    past_key_values = None

    # Initial decoder pass
    context_ids = torch.tensor(
        [decoder_context], dtype=torch.long, device=device
    )

    logits, past_key_values = model.decoder(
        context_ids,
        encoder_outputs,
        target_padding_mask=torch.ones_like(
            context_ids, dtype=torch.bool
        ),
        source_padding_mask=source_padding_mask,
        use_cache=True,
    )

    while word_count < max_new_words:

        next_logits = logits[:, -1, :].clone()

        # Prevent PAD and BOS generation
        next_logits[:, 0] = float("-inf")
        next_logits[:, 2] = float("-inf")

        if mode == "greedy":
            next_token = next_logits.argmax(
                dim=-1, keepdim=True
            )

        elif mode == "temperature":
            probs = torch.softmax(
                next_logits / temperature, dim=-1
            )
            next_token = torch.multinomial(probs, 1)

        else:  # top_k
            k = min(top_k, next_logits.size(-1))
            values, indices = torch.topk(
                next_logits / temperature, k, dim=-1
            )

            filtered = torch.full_like(
                next_logits, float("-inf")
            )
            filtered.scatter_(-1, indices, values)

            probs = torch.softmax(filtered, dim=-1)
            next_token = torch.multinomial(probs, 1)

        token_id = next_token.item()

        # EOS=3
        if token_id == 3:
            break

        generated.append(token_id)
        decoder_context.append(token_id)

        # Count completed BPE words
        if bpe.itos[token_id].endswith(b"</w>"):
            word_count += 1

        if word_count >= max_new_words:
            break

        # Rebuild cache when the context window is full
        cache_len = past_key_values[0][0][0].size(2)

        if cache_len >= max_seq_len:
            decoder_context = (
                [2] + decoder_context[-(max_seq_len - 1):]
            )

            context_ids = torch.tensor(
                [decoder_context],
                dtype=torch.long,
                device=device,
            )

            logits, past_key_values = model.decoder(
                context_ids,
                encoder_outputs,
                target_padding_mask=torch.ones_like(
                    context_ids, dtype=torch.bool
                ),
                source_padding_mask=source_padding_mask,
                past_key_values=None,
                use_cache=True,
            )

        else:
            # Process only the newest token
            logits, past_key_values = model.decoder(
                next_token,
                encoder_outputs,
                target_padding_mask=torch.ones(
                    (1, cache_len + 1),
                    dtype=torch.bool,
                    device=device,
                ),
                source_padding_mask=source_padding_mask,
                past_key_values=past_key_values,
                use_cache=True,
            )

    return bpe.decode(generated)


In [26]:
predictions = {}
count =0 
for  id, data in test_data.items():
    text =data["text"]
    input_ids = bpe.encode(text)
    print(id)
    print(f"  BPE length: {len(input_ids)}")
    if len(input_ids) == 0:
        print("EMPTY INPUT:", id)
        print("Text:", repr(text[:200]))
        continue
    predictions[id] = generate_tokens(model.module, text).replace("<s>", "").replace("</s>", "").replace("<pad>", "")
    count+=1
    if count>100:
        break

1
  BPE length: 674
2
  BPE length: 666
3
  BPE length: 586
4
  BPE length: 637
5
  BPE length: 612
6
  BPE length: 619
7
  BPE length: 661
8
  BPE length: 722
9
  BPE length: 646
10
  BPE length: 612
11
  BPE length: 771
12
  BPE length: 897
13
  BPE length: 610
14
  BPE length: 670
15
  BPE length: 671
16
  BPE length: 675
17
  BPE length: 645
18
  BPE length: 710
19
  BPE length: 782
20
  BPE length: 662
21
  BPE length: 594
22
  BPE length: 596
23
  BPE length: 593
24
  BPE length: 660
25
  BPE length: 586
26
  BPE length: 650
27
  BPE length: 642
28
  BPE length: 637
29
  BPE length: 688
30
  BPE length: 670
31
  BPE length: 744
32
  BPE length: 635
33
  BPE length: 604
34
  BPE length: 603
35
  BPE length: 588
36
  BPE length: 673
37
  BPE length: 705
38
  BPE length: 725
39
  BPE length: 769
40
  BPE length: 739
41
  BPE length: 706
42
  BPE length: 616
43
  BPE length: 612
44
  BPE length: 582
45
  BPE length: 599
46
  BPE length: 634
47
  BPE length: 667
48
  BPE length: 657
4

In [27]:
import json

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            id: {"text": text}
            for id, text in predictions.items()
        },
        f,
        ensure_ascii=False,
        indent=2
    )

In [28]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip
